# Trace the roads you can see
**SSTA Workshop 3** · Super-resolution use cases · Metro Manila

1. Press ▶ on the cell below (or click into it and press **Shift + Enter**). If Colab says the notebook was not authored by Google, choose **Run anyway**.
2. After a few seconds the exercise appears under the cell. Scroll so that it fills your screen, then press **Start tracing**.

You need a laptop and a Google account. Your lines are added, without your name, to the room's results.

In [ ]:
#@title ▶ Trace the roads { display-mode: "form" }
# SSTA Workshop 3: the tracing page of https://adb-data-division.github.io/ssta-trace/exercises/trace/, shown under this cell.
# Your browser only talks to Colab: this notebook's runtime (a Google server) downloads the images and
# sends your lines, without your name, to the room's results.
SESSION = "workshop"   # results group: the same as the web page's default (its ?s=), and as "session" below

import json, pathlib, random, re, string, time, urllib.parse, urllib.request
from IPython.display import HTML, JSON, display

def _ssta_get(name):
    """One file of the tracing site (images, road map, scoring, results link), downloaded once per runtime."""
    cache = pathlib.Path("/tmp/ssta_trace") / name
    if cache.exists():
        return cache.read_text(encoding="utf-8")
    err = None
    path = name if name == "config.js" else "exercises/trace/" + name      # where the file is in the repo
    for base in ("https://raw.githubusercontent.com/ADB-Data-Division/ssta-trace/main/",
                 "https://adb-data-division.github.io/ssta-trace/",
                 "https://cdn.jsdelivr.net/gh/ADB-Data-Division/ssta-trace@main/"):
        try:
            with urllib.request.urlopen(base + path, timeout=30) as r:
                text = r.read().decode("utf-8")
            cache.parent.mkdir(parents=True, exist_ok=True)
            cache.write_text(text, encoding="utf-8")
            return text
        except Exception as e:
            err = e
    raise RuntimeError(f"Could not download {name} of the tracing site: {err}")

_SITE = {n: _ssta_get(n) for n in ("areas.js", "score.js", "style.css", "config.js")}
ENDPOINT = (re.search(r'ENDPOINT\s*=\s*"([^"]*)"', _SITE["config.js"]) or [None, ""])[1]

try:
    from google.colab import output as _colab
except ImportError:  # outside Colab the page talks to the results sheet itself
    _colab = None

def _ssta_page(template, cfg):
    return HTML(template.replace("__CFG__", json.dumps(cfg)).replace("__STYLE__", _SITE["style.css"])
                .replace("__DATA__", _SITE["areas.js"] + "\n" + _SITE["score.js"]))

_UID = globals().get("_UID") or "colab-" + "".join(random.choices(string.ascii_lowercase + string.digits, k=8))
_SENT = globals().get("_SENT", 0)

def _ssta_send(body):
    """Called by the page when someone has traced both images: adds one row to the results sheet."""
    global _SENT
    if not ENDPOINT:
        return JSON({"ok": False, "error": "no results link in config.js"})
    d = json.loads(body)
    _SENT += 1
    d.update(session=SESSION, id=_UID, attempt=_SENT)
    req = urllib.request.Request(ENDPOINT, data=json.dumps(d).encode("utf-8"),
                                 headers={"Content-Type": "text/plain;charset=utf-8"})
    err = ""
    for k in range(3):
        try:
            with urllib.request.urlopen(req, timeout=30) as r:
                reply = r.read().decode("utf-8", "replace").strip()
            return JSON({"ok": reply == "ok", "error": "" if reply == "ok" else reply[:300]})
        except Exception as e:
            err = str(e)[:300]
            time.sleep(2 * (k + 1))
    return JSON({"ok": False, "error": err})

if _colab:
    _colab.register_callback("ssta_trace.send", _ssta_send)

# The page itself (HTML and JavaScript); the site's images, roads and styles go in at __DATA__ and __STYLE__.
_TRACE_PAGE = r'''<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Carlito:wght@400;700&family=Open+Sans:wght@700&display=swap">
<style>
__STYLE__
/* In Colab: the page sits in the cell's output, under the notebook's own toolbars. */
html, body { margin: 0; background: var(--page); }
body { font-size: 17px; }
.wrap { max-width: 1000px; margin: 0; padding: 16px 16px 28px; }
.intro { display: grid; grid-template-columns: minmax(0, 1fr) 240px; gap: 24px; align-items: start; }
.intro .stage { max-width: 240px; }
.trace { display: flex; flex-wrap: wrap; gap: 18px; align-items: flex-start; }
#tStage { width: min(var(--fit, 560px), 100%); flex: 0 0 auto; }
.side { flex: 1 1 210px; max-width: 300px; display: flex; flex-direction: column; gap: 12px; }
.side .step { font: 700 1.15rem var(--head); color: var(--title); }
.side .timer { flex: 0 0 auto; }
.side .btns { display: grid; grid-template-columns: 1fr 1fr; gap: 8px; }
.side .btns .go { grid-column: 1 / -1; }
@media (max-width: 620px) { .intro { grid-template-columns: 1fr; } }
</style>
<div class="wrap">
  <section id="intro">
    <div class="intro">
      <div style="display:flex;flex-direction:column;gap:14px">
        <div class="eyebrow"><b>SSTA Workshop 3</b> · Super-resolution use cases · Metro Manila</div>
        <h1>Trace the roads you can see</h1>
        <p>You get two satellite images of residential neighbourhoods in Metro Manila, each 500 m across. Draw a straight line along every road you can make out: press, drag and let go. Then we compare your lines with the real road map.</p>
        <ol class="steps">
          <li><span><b>Image 1:</b> 60 seconds. Draw a line along each road. For a bend, start a new line where the last one ended: it snaps on.</span></li>
          <li><span><b>Image 2:</b> another 60 seconds, a different neighbourhood.</span></li>
          <li><span><b>Your score:</b> how many of the roads you found on each image. Your lines are added, without your name, to the room's results.</span></li>
        </ol>
        <div class="bar"><button class="go" id="start">Start tracing</button><span class="muted">Lines don't have to be perfect. You can stop early with Done.</span></div>
      </div>
      <div class="stage s2"><img id="introImg" alt="Example: a Sentinel-2 image of Metro Manila, enlarged"></div>
    </div>
  </section>

  <section id="trace" hidden>
    <div class="trace">
      <div class="stage" id="tStage"><img id="tImg" alt=""><canvas id="tCan" width="600" height="600" aria-label="Drawing area: press, drag and release to draw a straight line along a road"></canvas>
        <div class="scale"><i></i><span>100 m</span></div></div>
      <div class="side">
        <span class="step" id="tStep">Image 1 of 2</span>
        <span class="chip" id="tChip" style="align-self:flex-start"></span>
        <div class="timer"><div class="track"><b id="tBar"></b></div><span id="tSec">60 s</span></div>
        <p class="muted">Press, drag and let go: one straight line along each road. For a bend, start the next line where the last one ended.</p>
        <div class="btns"><button id="undo">Undo</button><button id="clear">Clear</button><button class="go" id="done">Done</button></div>
      </div>
    </div>
  </section>

  <section id="result" hidden>
    <div style="display:flex;flex-direction:column;gap:16px">
      <div class="eyebrow"><b>Your score</b></div>
      <h2 id="rHead"></h2>
      <div class="cards">
        <div class="card s2"><span class="chip s2" style="align-self:flex-start">Sentinel-2 · 10 m</span><span class="big" id="rBlur"></span><span>of the roads found</span><span class="muted" id="rBlurP"></span></div>
        <div class="card sr"><span class="chip sr" style="align-self:flex-start">Super-resolved · 2.5 m</span><span class="big" id="rSharp"></span><span>of the roads found</span><span class="muted" id="rSharpP"></span></div>
      </div>
      <p class="small" id="sent"></p>
      <p>Both images come from the same free Sentinel-2 satellite. The second was super-resolved by our model before you traced it. Below, your lines are drawn on a real high-resolution image of each neighbourhood.</p>
      <div class="legend"><span style="--c:var(--yours)">your lines</span><span style="--c:var(--found)">roads you found</span><span style="--c:var(--missed)">roads you missed</span></div>
      <div class="pair">
        <figure><div class="stage real"><img id="rImg0" alt=""><canvas id="rCan0" width="600" height="600"></canvas></div><figcaption id="rCap0"></figcaption></figure>
        <figure><div class="stage real"><img id="rImg1" alt=""><canvas id="rCan1" width="600" height="600"></canvas></div><figcaption id="rCap1"></figcaption></figure>
      </div>
      <p class="small">A road counts as found when one of your lines runs within 12 m of it. Driveways and footpaths are left out. Roads: © OpenStreetMap contributors. Real images: Google satellite imagery, averaged to 2.5 m. Sentinel-2: contains modified Copernicus Sentinel data 2025.</p>
      <div class="bar"><button id="again">Trace again</button></div>
    </div>
  </section>
</div>
<script>
(() => {
__DATA__
const CFG = __CFG__;
const SECONDS = 60, SNAP = 14, MINLEN = 6;
let plan, cur, strokes = [], active = null, timer, left = 0, t0 = 0, results, attempt = 0;
const $ = id => document.getElementById(id);
const kernel = window.google && google.colab && google.colab.kernel;
try { google.colab.output.setIframeHeight(0, true, { maxHeight: 5000 }); } catch (e) {}
// Size the tracing image to the screen, so it fits under the browser's and Colab's toolbars.
const fit = Math.round(Math.max(320, Math.min(640, ((window.screen && screen.availHeight) || 900) - 300)));
document.documentElement.style.setProperty("--fit", fit + "px");
$("introImg").src = AREAS.A.img.blurry;

function newPlan() {
  const first = Math.random() < 0.5 ? "A" : "B";
  return [{ area: first, kind: "blurry" }, { area: first === "A" ? "B" : "A", kind: "sharp" }];
}
function show(id) {
  for (const s of ["intro", "trace", "result"]) $(s).hidden = s !== id;
  setTimeout(() => { try { $(id).scrollIntoView({ block: "nearest", behavior: "smooth" }); } catch (e) {} }, 300);
}
function begin(i) {
  cur = i; strokes = []; active = null;
  const p = plan[i], s2 = p.kind === "blurry";
  $("tStep").textContent = `Image ${i + 1} of 2`;
  $("tChip").textContent = s2 ? "Sentinel-2 · 10 m pixels, enlarged" : "Super-resolved · 2.5 m pixels";
  $("tChip").className = "chip " + (s2 ? "s2" : "sr");
  $("tStage").className = "stage " + (s2 ? "s2" : "sr");
  $("tImg").src = AREAS[p.area].img[p.kind]; $("tImg").alt = $("tChip").textContent;
  redraw(); t0 = Date.now(); left = SECONDS; tick();
  clearInterval(timer);   // clock time, so a throttled timer cannot stretch the minute
  timer = setInterval(() => { left = SECONDS - Math.floor((Date.now() - t0) / 1000); tick(); if (left <= 0) finish(); }, 250);
  show("trace");
}
function tick() { $("tSec").textContent = `${Math.max(left, 0)} s`; $("tBar").style.width = `${100 * Math.max(left, 0) / SECONDS}%`; }
function finish() {
  clearInterval(timer);
  if (active) { if (Math.hypot(active[1][0] - active[0][0], active[1][1] - active[0][1]) >= MINLEN) strokes.push(active); active = null; }
  plan[cur].strokes = strokes;
  if (cur === 0) begin(1); else score();
}
// Straight lines: press = start, drag = preview, release = end. Ends snap to the ends of earlier lines.
const can = $("tCan"), ctx = can.getContext("2d");
function pos(e) { const r = can.getBoundingClientRect(); return [(e.clientX - r.left) * 600 / r.width, (e.clientY - r.top) * 600 / r.height]; }
const clampP = p => [Math.min(600, Math.max(0, p[0])), Math.min(600, Math.max(0, p[1]))];
function snap(p) {
  let best = null, bd = SNAP;
  for (const s of strokes) for (const q of s) { const d = Math.hypot(p[0] - q[0], p[1] - q[1]); if (d < bd) { bd = d; best = q; } }
  return best ? [best[0], best[1]] : p;
}
can.addEventListener("pointerdown", e => { if (left <= 0) return; e.preventDefault(); can.setPointerCapture(e.pointerId);
  const p = snap(clampP(pos(e))); active = [p, p]; redraw(); });
can.addEventListener("pointermove", e => { if (!active) return; active[1] = clampP(pos(e)); redraw(); });
can.addEventListener("pointerup", () => { if (!active) return; const end = snap(active[1]);
  if (Math.hypot(end[0] - active[0][0], end[1] - active[0][1]) >= MINLEN) strokes.push([active[0], end]);
  active = null; redraw(); });
can.addEventListener("pointercancel", () => { active = null; redraw(); });
function redraw() {
  ctx.clearRect(0, 0, 600, 600);
  const all = strokes.concat(active ? [active] : []);
  for (const s of all) { polyline(ctx, s, "rgba(0,0,0,.6)", 8); polyline(ctx, s, css("--trace"), 4); }
  ctx.fillStyle = css("--trace"); ctx.strokeStyle = "rgba(0,0,0,.6)"; ctx.lineWidth = 2;
  for (const s of all) for (const q of s) { ctx.beginPath(); ctx.arc(q[0], q[1], 5, 0, 7); ctx.fill(); ctx.stroke(); }
}
$("undo").onclick = () => { strokes.pop(); redraw(); };
$("clear").onclick = () => { strokes = []; redraw(); };
$("done").onclick = finish;
$("start").onclick = $("again").onclick = () => { plan = newPlan(); attempt += 1; begin(0); };

const wait = ms => new Promise(r => setTimeout(r, ms));
async function send(rs) {
  const say = t => { $("sent").textContent = t; };
  const failed = "Could not reach the results board. Your score above still counts.";
  const plan = rs.map(r => ({ area: r.area, kind: r.kind,
    recall: +r.recall.toFixed(4), precision: r.precision === null ? null : +r.precision.toFixed(4),
    lines: r.strokes.map(s => s.flat().map(Math.round)) }));
  say("Sending your lines to the room…");
  if (kernel) {   // in Colab: the runtime (a Google server) posts the lines to the results sheet
    for (let k = 0; k < 3; k++) {
      try {
        const res = await google.colab.kernel.invokeFunction("ssta_trace.send", [JSON.stringify({ attempt, plan })], {});
        const d = res && res.data && res.data["application/json"];
        if (d && d.ok) return say("Your lines were added to the room's results.");
        return say(failed);
      } catch (e) { await wait(3000 * (k + 1)); }   // runtime busy or reconnecting
    }
    return say(failed + " (Is the notebook still connected? See the top right of the page.)");
  }
  if (!CFG.endpoint) return say("");
  const body = JSON.stringify({ session: CFG.session, id: CFG.uid, attempt, plan });
  for (let k = 0; k < 3; k++) {
    try {
      await fetch(CFG.endpoint, { method: "POST", mode: "no-cors", headers: { "Content-Type": "text/plain;charset=utf-8" }, body, keepalive: true });
      return say("Your lines were added to the room's results.");
    } catch (e) { await wait(2000 * (k + 1)); }
  }
  say(failed);
}
function score() {
  results = plan.map(p => ({ ...p, ...evaluate(p.area, p.strokes) }));
  const [b, s] = results;
  $("rBlur").textContent = pct(b.recall); $("rSharp").textContent = pct(s.recall);
  const prec = r => r.precision === null ? "You drew no lines." : `${pct(r.precision)} of your lines were on a real road.`;
  $("rBlurP").textContent = prec(b); $("rSharpP").textContent = prec(s);
  const ratio = b.recall > 0.005 ? s.recall / b.recall : null;
  $("rHead").textContent = s.recall > b.recall
    ? (ratio && ratio >= 1.5 ? `You found ${ratio.toFixed(1)} times as many roads on the super-resolved image` : "You found more roads on the super-resolved image")
    : "You found as many roads on the blurry image. Try again, slowly?";
  results.forEach((r, i) => {
    $("rImg" + i).src = AREAS[r.area].img.real; $("rImg" + i).alt = "Real high-resolution image of the same neighbourhood";
    $("rCap" + i).innerHTML = `<b>Traced on ${r.kind === "blurry" ? "Sentinel-2" : "the super-resolved image"}</b> · drawn on the real image`;
    const c = $("rCan" + i).getContext("2d"); c.clearRect(0, 0, 600, 600);
    const shown = r.segs.filter(([, , ok, m]) => m || ok);   // outlines first, then colours, so the pieces join up
    for (const [p, q, ok] of shown) polyline(c, [p, q], "rgba(0,0,0,.45)", ok ? 9 : 5);
    for (const [p, q, ok] of shown) polyline(c, [p, q], ok ? css("--found") : css("--missed"), ok ? 6 : 3);
    for (const st of r.strokes) { polyline(c, st, "rgba(0,0,0,.6)", 5); polyline(c, st, css("--yours"), 2.5); }
  });
  show("result"); send(results);
}
})();
</script>
'''

display(_ssta_page(_TRACE_PAGE, {"session": SESSION, "endpoint": ENDPOINT, "uid": _UID}))


## The room's results (presenter)
Run the cell below on the presenter's screen to show everyone's results; it updates every 5 seconds. Participants don't need to run it (it would ask the results sheet for updates from every laptop).

To check the layout before anyone has traced, tick `simulated_people`. For a rehearsal, change `session` here and `SESSION` in the code of the cell above (*Show code*) to the same new name.

In [ ]:
#@title Room results { display-mode: "form" }
session = "workshop"  #@param {type:"string"}
simulated_people = False  #@param {type:"boolean"}
# Everyone's results, from the web page and from this notebook, updated every 5 seconds.
# The runtime reads the results sheet, so this also works on networks that block github.io.

import json, pathlib, random, re, string, time, urllib.parse, urllib.request
from IPython.display import HTML, JSON, display

def _ssta_get(name):
    """One file of the tracing site (images, road map, scoring, results link), downloaded once per runtime."""
    cache = pathlib.Path("/tmp/ssta_trace") / name
    if cache.exists():
        return cache.read_text(encoding="utf-8")
    err = None
    path = name if name == "config.js" else "exercises/trace/" + name      # where the file is in the repo
    for base in ("https://raw.githubusercontent.com/ADB-Data-Division/ssta-trace/main/",
                 "https://adb-data-division.github.io/ssta-trace/",
                 "https://cdn.jsdelivr.net/gh/ADB-Data-Division/ssta-trace@main/"):
        try:
            with urllib.request.urlopen(base + path, timeout=30) as r:
                text = r.read().decode("utf-8")
            cache.parent.mkdir(parents=True, exist_ok=True)
            cache.write_text(text, encoding="utf-8")
            return text
        except Exception as e:
            err = e
    raise RuntimeError(f"Could not download {name} of the tracing site: {err}")

_SITE = {n: _ssta_get(n) for n in ("areas.js", "score.js", "style.css", "config.js")}
ENDPOINT = (re.search(r'ENDPOINT\s*=\s*"([^"]*)"', _SITE["config.js"]) or [None, ""])[1]

try:
    from google.colab import output as _colab
except ImportError:  # outside Colab the page talks to the results sheet itself
    _colab = None

def _ssta_page(template, cfg):
    return HTML(template.replace("__CFG__", json.dumps(cfg)).replace("__STYLE__", _SITE["style.css"])
                .replace("__DATA__", _SITE["areas.js"] + "\n" + _SITE["score.js"]))

SESSION = re.sub(r"[^\w-]", "", session)[:40] or "workshop"

def _ssta_rows(sess):
    """Called by the page every 5 s: all rows of the session, oldest first."""
    try:
        url = ENDPOINT + "?" + urllib.parse.urlencode({"session": sess, "t": int(time.time() * 1000)})
        with urllib.request.urlopen(url, timeout=30) as r:
            return JSON({"rows": json.loads(r.read().decode("utf-8"))})
    except Exception as e:
        return JSON({"error": str(e)[:300]})

if _colab:
    _colab.register_callback("ssta_trace.rows", _ssta_rows)

# The page itself (HTML and JavaScript); the site's images, roads and styles go in at __DATA__ and __STYLE__.
_RESULTS_PAGE = r'''<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Carlito:wght@400;700&family=Open+Sans:wght@700&display=swap">
<style>
__STYLE__
html, body { margin: 0; background: var(--page); }
.wrap.wide { margin: 0; padding: 16px 16px 28px; }
.top { display: grid; grid-template-columns: minmax(0, 1fr) minmax(0, 1fr) minmax(0, 1.3fr); gap: 14px; }
.card .big { font-size: 3.4rem; }
.maps { display: grid; grid-template-columns: auto minmax(0, 1fr) minmax(0, 1fr); gap: 10px 14px; align-items: center; }
.maps .rowlab { writing-mode: vertical-rl; transform: rotate(180deg); font: 700 .95rem var(--head); color: var(--muted); }
.maps .collab { text-align: center; }
.stage.map { max-width: none; }
svg text { font-family: var(--body); }
.status { display: flex; flex-wrap: wrap; gap: 12px; align-items: center; }
.demo { background: var(--trace); color: #000; font-weight: 700; padding: 2px 10px; border-radius: 6px; }
label { display: inline-flex; gap: 6px; align-items: center; color: var(--muted); }
@media (max-width: 900px) { .top { grid-template-columns: 1fr 1fr; } .top .slope { grid-column: 1 / -1; } }
@media (max-width: 560px) { .top { grid-template-columns: 1fr; } .maps { grid-template-columns: minmax(0, 1fr) minmax(0, 1fr); } .maps .rowlab, .maps .sp { display: none; } }
</style>
<div class="wrap wide">
  <div class="status"><span class="eyebrow"><b>SSTA Workshop 3</b> · Trace the roads · room results</span><span class="demo" id="demo" hidden>Demo data</span>
    <span class="small" id="stamp"></span></div>
  <h1 id="head">Waiting for the first results…</h1>
  <div class="top">
    <div class="card s2"><span class="chip s2" style="align-self:flex-start">Traced on Sentinel-2 · 10 m</span><span class="big" id="mS2">–</span><span>of the roads found (median)</span></div>
    <div class="card sr"><span class="chip sr" style="align-self:flex-start">Traced on super-resolved · 2.5 m</span><span class="big" id="mSR">–</span><span>of the roads found (median)</span></div>
    <div class="card slope"><h3 id="slopeHead">Each line is one person</h3><svg id="slope" viewBox="0 0 420 190" role="img" aria-label="Share of roads found by each person on Sentinel-2 and on the super-resolved image"></svg></div>
  </div>
  <div class="bar"><h2 style="flex:1 1 auto">Everyone's lines on the real image</h2>
    <label><input type="checkbox" id="osm"> show the road map</label></div>
  <div class="maps" id="maps">
    <span class="sp"></span><div class="collab"><span class="chip s2">Lines drawn on Sentinel-2</span></div><div class="collab"><span class="chip sr">Lines drawn on the super-resolved image</span></div>
  </div>
  <p class="small">Each person traced one neighbourhood on Sentinel-2 and the other on the super-resolved image, assigned at random. A road counts as found when a line runs within 12 m of it. Latest attempt per device; people who used the web page and the notebook are counted together. Roads: © OpenStreetMap contributors. Real images: Google satellite imagery, averaged to 2.5 m.</p>
</div>
<script>
(() => {
__DATA__
const CFG = __CFG__;
const $ = id => document.getElementById(id);
const kernel = window.google && google.colab && google.colab.kernel;
try { google.colab.output.setIframeHeight(0, true, { maxHeight: 5000 }); } catch (e) {}
const DEMO = CFG.demo || !CFG.endpoint;
const AREA_NAME = { A: "Neighbourhood 1", B: "Neighbourhood 2" };
const cans = {};
for (const a of ["A", "B"]) {
  const lab = document.createElement("span"); lab.className = "rowlab"; lab.textContent = AREA_NAME[a]; $("maps").appendChild(lab);
  for (const k of ["blurry", "sharp"]) {
    const f = document.createElement("figure");
    f.innerHTML = `<div class="stage real map"><img alt="Real high-resolution image, ${AREA_NAME[a]}" src="${AREAS[a].img.real}"><canvas width="600" height="600"></canvas></div><figcaption></figcaption>`;
    $("maps").appendChild(f); cans[a + k] = { c: f.querySelector("canvas"), cap: f.querySelector("figcaption") };
  }
}
function demoRows(n = 24) {   // simulated room: road pieces found with lower odds on Sentinel-2
  const rows = [];
  for (let i = 0; i < n; i++) {
    const first = i % 2 ? "A" : "B", plan = [{ area: first, kind: "blurry" }, { area: first === "A" ? "B" : "A", kind: "sharp" }];
    for (const p of plan) {
      const odds = p.kind === "blurry" ? 0.08 + 0.12 * Math.random() : 0.35 + 0.3 * Math.random();
      const j = () => (Math.random() - .5) * (p.kind === "blurry" ? 22 : 8);
      p.lines = AREAS[p.area].roads.filter(r => r.m && Math.random() < odds).map(r => [r.p[0][0] + j(), r.p[0][1] + j(), r.p.at(-1)[0] + j(), r.p.at(-1)[1] + j()]);
      if (p.kind === "blurry") for (let k = 0; k < 4; k++) p.lines.push([600 * Math.random(), 600 * Math.random(), 600 * Math.random(), 600 * Math.random()].map(Math.round));
    }
    rows.push({ id: "demo" + i, attempt: 1, plan });
  }
  return rows;
}
let rows = [];
async function load() {
  if (DEMO) { if (!rows.length) rows = demoRows(); return; }
  if (kernel) {   // in Colab: the runtime reads the results sheet
    const res = await google.colab.kernel.invokeFunction("ssta_trace.rows", [CFG.session], {});
    const d = res.data["application/json"];
    if (d.error) throw new Error(d.error);
    rows = d.rows; return;
  }
  const r = await fetch(`${CFG.endpoint}?session=${encodeURIComponent(CFG.session)}&t=${Date.now()}`);
  rows = await r.json();
}
const median = a => { if (!a.length) return null; const s = [...a].sort((x, y) => x - y), m = s.length >> 1; return s.length % 2 ? s[m] : (s[m - 1] + s[m]) / 2; };
function render() {
  const latest = new Map(); for (const r of rows) latest.set(r.id, r);         // rows arrive in time order: keep each device's last attempt
  const people = [...latest.values()].filter(r => Array.isArray(r.plan) && r.plan.length === 2).map(r => {
    const p = r.plan.map(q => { const lines = (q.lines || []).map(l => [[l[0], l[1]], [l[2], l[3]]]); return { ...q, lines, recall: evaluate(q.area, lines).recall }; });
    return { b: p.find(q => q.kind === "blurry"), s: p.find(q => q.kind === "sharp") };
  }).filter(x => x.b && x.s);
  const n = people.length;
  $("demo").hidden = !DEMO;
  $("stamp").textContent = DEMO ? (CFG.endpoint ? "Preview with simulated people. Untick simulated_people for the real room." : "Preview with simulated people: the site has no results link (config.js).")
    : `Session "${CFG.session}" · updated ${new Date().toLocaleTimeString()}`;
  if (!n) { $("head").textContent = "Waiting for the first results…"; return; }
  const mb = median(people.map(x => x.b.recall)), ms = median(people.map(x => x.s.recall));
  const more = people.filter(x => x.s.recall > x.b.recall).length;
  $("mS2").textContent = pct(mb); $("mSR").textContent = pct(ms);
  $("head").textContent = mb > 0.005 && ms / mb >= 1.5
    ? `${n} people traced: they found ${(ms / mb).toFixed(1)} times as many roads on the super-resolved image`
    : `${n} people traced: ${more} of them found more roads on the super-resolved image`;
  $("slopeHead").textContent = `${more} of ${n} found more roads on the super-resolved image`;
  // slope chart
  const x0 = 110, x1 = 310, y = v => 170 - 150 * v, g = [];
  for (const v of [0, .5, 1]) g.push(`<line x1="${x0 - 10}" x2="${x1 + 10}" y1="${y(v)}" y2="${y(v)}" stroke="#e6e6e6"/><text x="${x0 - 16}" y="${y(v) + 5}" text-anchor="end" font-size="14" fill="#737373">${v * 100}%</text>`);
  g.push(`<text x="${x0}" y="188" text-anchor="middle" font-size="14" fill="#737373">Sentinel-2</text><text x="${x1}" y="188" text-anchor="middle" font-size="14" fill="#1f8a70">Super-resolved</text>`);
  for (const p of people) g.push(`<line x1="${x0}" y1="${y(p.b.recall)}" x2="${x1}" y2="${y(p.s.recall)}" stroke="${p.s.recall > p.b.recall ? "#1f8a70" : "#bfbfbf"}" stroke-opacity=".55" stroke-width="2"/>`
    + `<circle cx="${x0}" cy="${y(p.b.recall)}" r="4" fill="#737373"/><circle cx="${x1}" cy="${y(p.s.recall)}" r="4" fill="#1f8a70"/>`);
  g.push(`<line x1="${x0}" y1="${y(mb)}" x2="${x1}" y2="${y(ms)}" stroke="#007db7" stroke-width="5"/><text x="${x1 + 14}" y="${y(ms) + 5}" font-size="14" font-weight="700" fill="#007db7">median</text>`);
  $("slope").innerHTML = g.join("");
  // overlays
  const showRoads = $("osm").checked;
  for (const a of ["A", "B"]) for (const k of ["blurry", "sharp"]) {
    const { c, cap } = cans[a + k], ctx = c.getContext("2d"); ctx.clearRect(0, 0, 600, 600);
    if (showRoads) for (const r of AREAS[a].roads) if (r.m) polyline(ctx, r.p, "rgba(233,83,43,.85)", 3);
    const here = people.map(x => x.b.area === a && k === "blurry" ? x.b : x.s.area === a && k === "sharp" ? x.s : null).filter(Boolean);
    const alpha = Math.max(.12, Math.min(.6, 3 / Math.max(1, here.length)));
    for (const q of here) for (const l of q.lines) { polyline(ctx, l, `rgba(0,0,0,${alpha * .6})`, 6); polyline(ctx, l, `rgba(253,181,21,${alpha + .2})`, 3); }
    cap.innerHTML = here.length ? `<b>${here.length} people</b> · median ${pct(median(here.map(q => q.recall)))} of the roads found` : "nobody yet";
  }
}
$("osm").onchange = render;
async function cycle() {
  try { await load(); render(); } catch (e) { $("stamp").textContent = "Could not load results, retrying…"; }
  if (!DEMO) setTimeout(cycle, 5000);
}
cycle();
})();
</script>
'''

display(_ssta_page(_RESULTS_PAGE, {"session": SESSION, "endpoint": ENDPOINT, "demo": bool(simulated_people)}))
